## MIA loss threshold

In [1]:
# run_mia_loss_threshold.py
"""
Loss-threshold Membership Inference Attack (F1 selection only)

- Computes per-sample cross-entropy loss on a validation set with known membership labels.
- Selects a single global threshold tau that maximizes F1 on the validation set.
- Applies that threshold to an evaluation set and saves predictions.

"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score

# -------------------------
# Config (edit as needed)
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

victim_dir = "victim_model_distilbert_agnews"
validation_csv = "validation_samples.csv"       # must contain columns: id,text,label
validation_members = "validation_results.txt"   # lines: "<id> <member>" where member is 0/1
eval_csv = "sampled.csv"                        # must contain columns: id,text,(label optional)
output_path = "mia_lm_results.txt"

batch_size = 32
max_length = 256
n_threshold_candidates = 2000  # resolution for searching tau

# -------------------------
# Helpers
# -------------------------
def read_membership_file(path):
    mapping = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            _id = parts[0]
            try:
                member = int(parts[1])
            except ValueError:
                member = int(parts[-1])
            mapping[_id] = int(member)
    return mapping

def batchify(df, batch_size):
    for i in range(0, len(df), batch_size):
        yield df.iloc[i:i+batch_size]

@torch.no_grad()
def compute_losses(model, tokenizer, df, text_col="text", label_col="label", batch_size=32, max_length=256):
    """
    Compute per-sample cross-entropy loss using true labels (required).
    Returns numpy array of losses in the same order as df.
    """
    if label_col not in df.columns:
        raise ValueError(f"Label column '{label_col}' not found in dataframe. True labels are required.")
    losses = []
    model.eval()
    for batch in batchify(df, batch_size):
        texts = batch[text_col].astype(str).tolist()
        labels = batch[label_col].astype(int).tolist()
        enc = tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)
        labels_t = torch.tensor(labels, dtype=torch.long, device=device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        per_sample_loss = F.cross_entropy(logits, labels_t, reduction="none")
        losses.extend(per_sample_loss.detach().cpu().numpy().tolist())
    return np.array(losses)

def report_confusion(y_true, y_pred, label):
    """
    Print confusion counts and common metrics.
    Output format:
      <label> -> TP:<tp> FP:<fp> TN:<tn> FN:<fn>  predicted_members:<sum_pred>  precision:<p:.3f> recall:<r:.3f> f1:<f1:.3f>
    """
    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    except Exception:
        # handle degenerate cases
        tn = fp = fn = tp = 0
        unique_true = np.unique(y_true)
        if len(unique_true) == 1:
            if unique_true[0] == 0:
                tn = int((y_pred == 0).sum())
                fp = int((y_pred == 1).sum())
            else:
                tp = int((y_pred == 1).sum())
                fn = int((y_pred == 0).sum())

    # compute precision, recall, f1 robustly
    if (tp + fp) > 0:
        precision = tp / (tp + fp)
    else:
        precision = 0.0
    if (tp + fn) > 0:
        recall = tp / (tp + fn)
    else:
        recall = 0.0
    if (precision + recall) > 0:
        f1 = 2 * (precision * recall) / (precision + recall)
    else:
        f1 = 0.0

    print(f"{label} -> TP:{tp} FP:{fp} TN:{tn} FN:{fn}  predicted_members:{int(y_pred.sum())}  precision:{precision:.3f} recall:{recall:.3f} f1:{f1:.3f}")


# -------------------------
# Load model and tokenizer
# -------------------------
print("Loading model from:", victim_dir)
tokenizer = AutoTokenizer.from_pretrained(victim_dir, local_files_only=False)
model = AutoModelForSequenceClassification.from_pretrained(victim_dir, local_files_only=False)
model.to(device)
model.eval()

# -------------------------
# Load validation and membership labels
# -------------------------
print("Reading validation CSV:", validation_csv)
val_df = pd.read_csv(validation_csv, dtype=str)
if "id" not in val_df.columns:
    val_df.insert(0, "id", val_df.index.astype(str))
if "text" not in val_df.columns:
    raise ValueError("validation CSV must contain a 'text' column.")
if "label" not in val_df.columns:
    raise ValueError("validation CSV must contain a 'label' column with integer class ids.")
val_df["label"] = val_df["label"].astype(int)

print("Reading validation membership labels:", validation_members)
val_members_map = read_membership_file(validation_members)
val_df["member"] = val_df["id"].map(val_members_map)
if val_df["member"].isna().any():
    missing = val_df[val_df["member"].isna()]["id"].tolist()
    raise ValueError(f"Missing membership labels for validation ids: {missing[:20]}")
val_df["member"] = val_df["member"].astype(int)

# -------------------------
# Compute validation losses
# -------------------------
print("Computing validation losses...")
val_losses = compute_losses(model, tokenizer, val_df, text_col="text", label_col="label",
                            batch_size=batch_size, max_length=max_length)
val_df["loss"] = val_losses

# Diagnostics
print("Validation size:", len(val_df))
print("Member counts:", val_df["member"].value_counts().to_dict())
print("Loss stats min,25,median,75,max ->",
      np.min(val_losses), np.percentile(val_losses,25), np.median(val_losses),
      np.percentile(val_losses,75), np.max(val_losses))
print("Unique loss values:", len(np.unique(val_losses)))

y_val = val_df["member"].values
losses = val_df["loss"].values

# -------------------------
# F1-based threshold selection (global)
# -------------------------
print("\nSelecting threshold by maximizing F1 on validation...")
if len(np.unique(y_val)) == 1:
    # degenerate case: all members or all non-members
    if y_val[0] == 1:
        best_tau_f1 = losses.min() - 1e-6
        best_f1 = 1.0
    else:
        best_tau_f1 = losses.max() + 1e-6
        best_f1 = 0.0
else:
    candidates = np.linspace(losses.min() - 1e-6, losses.max() + 1e-6, n_threshold_candidates)
    best_tau_f1 = None
    best_f1 = -1.0
    for tau in candidates:
        pred = (losses < tau).astype(int)
        f1 = f1_score(y_val, pred)
        if f1 > best_f1:
            best_f1 = f1
            best_tau_f1 = float(tau)

# Evaluate current direction (lower loss => member)
pred_lower = (losses < best_tau_f1).astype(int)
f1_lower = f1_score(y_val, pred_lower)

# Evaluate flipped direction (higher loss => member)
pred_higher = (losses > best_tau_f1).astype(int)
f1_higher = f1_score(y_val, pred_higher)

print("F1 lower_loss_is_member:", f1_lower, "F1 higher_loss_is_member:", f1_higher)

# Choose the better direction
if f1_higher > f1_lower:
    print("Using flipped rule: higher loss => member")
    best_direction = "higher"
    pred_f1 = pred_higher
else:
    print("Using default rule: lower loss => member")
    best_direction = "lower"
    pred_f1 = pred_lower
report_confusion(y_val, pred_f1, "F1 (validation)")

# Also report ROC AUC (loss as score)
try:
    auc = roc_auc_score(y_val, -losses)
except Exception:
    auc = float("nan")
print("Validation ROC AUC (loss as score):", auc)

# -------------------------
# Apply to evaluation set
# -------------------------
print("\nReading evaluation CSV:", eval_csv)
eval_df = pd.read_csv(eval_csv, dtype=str)
if "id" not in eval_df.columns:
    eval_df.insert(0, "id", eval_df.index.astype(str))
if "text" not in eval_df.columns:
    raise ValueError("evaluation CSV must contain a 'text' column.")

has_label = ("label" in eval_df.columns) and (eval_df["label"].notna().all())
if has_label:
    eval_df["label"] = eval_df["label"].astype(int)
    eval_losses = compute_losses(model, tokenizer, eval_df, text_col="text", label_col="label",
                                 batch_size=batch_size, max_length=max_length)

eval_df["loss"] = eval_losses
eval_df["pred_f1"] = (eval_df["loss"].values < best_tau_f1).astype(int)

# If evaluation has true membership labels, compute metrics
if "member" in eval_df.columns and eval_df["member"].notna().all():
    eval_df["member"] = eval_df["member"].astype(int)
    print("\nEvaluation metrics (using F1 threshold):")
    report_confusion(eval_df["member"].values, eval_df["pred_f1"].values, "F1 (evaluation)")
    try:
        print("Eval ROC AUC (loss as score):", roc_auc_score(eval_df["member"].values, -eval_df["loss"].values))
    except Exception:
        pass

# -------------------------
# Save results
# -------------------------
with open(output_path, "w", encoding="utf-8") as f:
    for _, row in eval_df.iterrows():
        f.write(f"{row['id']} {int(row['pred_f1'])}\n")

print(f"\nSaved MIA results to: {output_path}")

# -------------------------
# Summary
# -------------------------
print("\nSummary:")
print(f"  Validation size: {len(val_df)}")
print(f"  Eval size: {len(eval_df)}")
print(f"  Selected F1 threshold (tau): {best_tau_f1}")
print(f"  Validation F1 (thresholded): {best_f1}")
print(f"  Validation ROC AUC (loss as score): {auc}")

C:\Users\Kevin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Loading model from: victim_model_distilbert_agnews
Reading validation CSV: validation_samples.csv
Reading validation membership labels: validation_results.txt
Computing validation losses...
Validation size: 1000
Member counts: {0: 808, 1: 192}
Loss stats min,25,median,75,max -> 0.017249442636966705 0.02811246132478118 0.0477499533444643 0.11590582318603992 5.162142276763916
Unique loss values: 1000

Selecting threshold by maximizing F1 on validation...
F1 lower_loss_is_member: 0.33166833166833165 F1 higher_loss_is_member: 0.13577023498694518
Using default rule: lower loss => member
F1 (validation) -> TP:166 FP:643 TN:165 FN:26  predicted_members:809  precision:0.205 recall:0.865 f1:0.332
Validation ROC AUC (loss as score): 0.5210009282178218

Reading evaluation CSV: sampled.csv

Saved MIA results to: mia_lm_results.txt

Summary:
  Validation size: 1000
  Eval size: 10000
  Selected F1 threshold (tau): 0.17167249968930137
  Validation F1 (thresholded): 0.33166833166833165
 

In [2]:
with open("mia_threshold_results.txt","r") as f:
    lines = [l.strip() for l in f if l.strip()]
print("Total lines in file:", len(lines))
print("First 20 lines:")
print("\n".join(lines[:20]))

ones = sum(1 for l in lines if l.split()[1] == '1')
zeros = len(lines) - ones
print(f"Predicted members (1): {ones}")
print(f"Predicted non-members (0): {zeros}")
print(f"Member ratio: {ones/len(lines):.4f}")

Total lines in file: 10000
First 20 lines:
0 1
1 0
2 1
3 1
4 1
5 1
6 1
7 1
8 1
9 1
10 0
11 1
12 0
13 1
14 1
15 1
16 1
17 1
18 1
19 1
Predicted members (1): 8080
Predicted non-members (0): 1920
Member ratio: 0.8080


## Shadow model attack

In [3]:
# save as run_mia_with_shadows.py
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import random
import os
from sklearn.metrics import (
    confusion_matrix, f1_score, accuracy_score,
    precision_score, recall_score, roc_auc_score
)


# -----------------------------
# SETUP
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Paths (adapt if needed)
victim_dir = "victim_model_distilbert_agnews"
validation_csv = "validation_samples.csv"       # has columns "text","label"
validation_members = "validation_results.txt"   # lines: "<id> <member>"
shadow_pool = "sampled.csv"                     # pool for shadow models
output_path = "mia_lm_results.txt"

# -----------------------------
# LOAD MODEL & TOKENIZER
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(victim_dir, local_files_only=False)
model = AutoModelForSequenceClassification.from_pretrained(victim_dir, local_files_only=False)
model.to(device)
model.eval()

# -----------------------------
# LOAD VALIDATION DATA
# -----------------------------
val_df = pd.read_csv(validation_csv)
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].astype(int).tolist()

val_members = {}
with open(validation_members, "r") as f:
    for line in f:
        idx, m = line.strip().split()
        val_members[int(idx)] = int(m)
val_member_labels = [val_members.get(i, 0) for i in range(len(val_df))]

print(f"\nValidation set: {sum(val_member_labels)} members out of {len(val_member_labels)} total")
print(f"Member ratio: {sum(val_member_labels)/len(val_member_labels):.2%}")

# -----------------------------
# IMPROVED SCORE COMPUTATION
# -----------------------------
def compute_scores(model, texts, labels=None, batch_size=32):
    model.eval()
    all_maxp, all_neglog_true, all_entropy, all_margin = [], [], [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=256).to(device)
        with torch.no_grad():
            out = model(**enc)
            logits = out.logits
            probs = F.softmax(logits, dim=-1)
            log_probs = torch.log(probs + 1e-12)
            
            # Max probability
            all_maxp.extend(probs.max(dim=1).values.cpu().numpy())
            
            # Entropy
            entropy = -(probs * log_probs).sum(dim=1)
            all_entropy.extend(entropy.cpu().numpy())
            
            # Margin (difference between top-2 predictions)
            top2 = torch.topk(probs, 2, dim=1).values
            margin = (top2[:, 0] - top2[:, 1])
            all_margin.extend(margin.cpu().numpy())
            
            if labels is not None:
                true_labels = torch.tensor(labels[i:i+len(batch)], device=device)
                # negative log-likelihood (positive values)
                all_neglog_true.extend((-log_probs[range(len(true_labels)), true_labels]).cpu().numpy())
    
    return {
        'max_prob': np.array(all_maxp),
        'neg_log_loss': np.array(all_neglog_true) if labels is not None else None,
        'entropy': np.array(all_entropy),
        'margin': np.array(all_margin)
    }

# -----------------------------
# BASELINE LOSS-THRESHOLD ATTACK
# -----------------------------
print("\nComputing victim model scores on validation set...")
val_scores = compute_scores(model, val_texts, val_labels)
# Use negative loss so that higher score => more likely member
val_loss_score = -val_scores['neg_log_loss']

fpr, tpr, thr = roc_curve(val_member_labels, val_loss_score)
youden = tpr - fpr
best_thr = thr[np.argmax(youden)]
auc_val = roc_auc_score(val_member_labels, val_loss_score)
preds_val = (val_loss_score >= best_thr).astype(int)
print(f"\nBaseline Loss-threshold attack:")
print(f"AUC: {auc_val:.4f}")
print(f"F1: {f1_score(val_member_labels, preds_val):.4f}, "
      f"Precision: {precision_score(val_member_labels, preds_val):.4f}, "
      f"Recall: {recall_score(val_member_labels, preds_val):.4f}")
print(f"Chosen threshold: {best_thr:.4f}")

# -----------------------------
# SHADOW MODEL TRAINING (disjoint sampling)
# -----------------------------
train_shadows = True
n_shadows = 3
shadow_frac = 0.3
epochs = 2
lr = 5e-5

if train_shadows:
    print("\n--- Training shadow models ---")
    pool_df = pd.read_csv(shadow_pool)
    pool_texts = pool_df["text"].tolist()
    pool_labels = pool_df["label"].astype(int).tolist()
    n_pool = len(pool_df)
    shadow_features, shadow_labels = [], []

    # Shuffle and partition pool indices into disjoint chunks
    all_indices = list(range(n_pool))
    random.shuffle(all_indices)
    chunk_size = n_pool // n_shadows
    shadow_chunks = [all_indices[i*chunk_size:(i+1)*chunk_size] for i in range(n_shadows)]

    for sid, chunk in enumerate(shadow_chunks):
        print(f"\n[Shadow {sid+1}/{n_shadows}]")

        # sample members from this chunk
        n_members = int(max(1, round(shadow_frac * len(chunk))))
        member_idx = random.sample(chunk, n_members)

        # nonmembers = remaining in this chunk
        remaining = [i for i in chunk if i not in member_idx]
        n_nonmembers = min(len(remaining), n_members)
        nonmember_idx = random.sample(remaining, n_nonmembers)

        train_texts = [pool_texts[i] for i in member_idx]
        train_labels = [pool_labels[i] for i in member_idx]

        nonmember_texts = [pool_texts[i] for i in nonmember_idx]
        nonmember_labels = [pool_labels[i] for i in nonmember_idx]

        # fine-tune a shadow model (start from victim weights)
        shadow_model = AutoModelForSequenceClassification.from_pretrained(victim_dir).to(device)

        shadow_ds = Dataset.from_dict({"text": train_texts, "label": train_labels})
        def tokenize_fn(ex):
            return tokenizer(ex["text"], padding="max_length", truncation=True, max_length=128)
        shadow_ds = shadow_ds.map(tokenize_fn, batched=True)
        shadow_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

        args = TrainingArguments(
            output_dir=f"shadow_{sid}",
            per_device_train_batch_size=8,
            num_train_epochs=epochs,
            learning_rate=lr,
            logging_steps=50,
            save_strategy="no",
            report_to="none"
        )
        trainer = Trainer(model=shadow_model, args=args, train_dataset=shadow_ds)
        trainer.train()

        # Collect features for member/non-member
        for subset, texts, labs, m_flag in [
            ("members", train_texts, train_labels, 1),
            ("nonmembers", nonmember_texts, nonmember_labels, 0)
        ]:
            if len(texts) == 0:
                continue
            scores = compute_scores(shadow_model, texts, labs)
            feats = np.stack([
                scores['max_prob'],
                -scores['neg_log_loss'],
                scores['entropy'],
                scores['margin']
            ], axis=1)
            shadow_features.append(feats)
            shadow_labels.append(np.full(len(feats), m_flag))

        print(f" Shadow {sid}: members={len(train_texts)}, nonmembers={len(nonmember_texts)}")

    # combine and train meta-classifier
    X = np.vstack(shadow_features)
    y = np.concatenate(shadow_labels)

    print(f"\nShadow training stats:")
    print(f"Total samples: {len(y)}, Members: {sum(y)}, Non-members: {len(y)-sum(y)}")
    print(f"Feature shape: {X.shape}")

    scaler = StandardScaler().fit(X)
    X = scaler.transform(X)

    # Use class_weight='balanced' to handle imbalanced validation set
    meta = LogisticRegression(max_iter=200, class_weight='balanced', random_state=SEED)
    meta.fit(X, y)
    print("Meta-classifier trained on shadow data.")

    # evaluate meta on validation set (using victim outputs)
    print("\nEvaluating meta-classifier on validation set...")
    val_feats = np.stack([
        val_scores['max_prob'],
        val_loss_score,
        val_scores['entropy'],
        val_scores['margin']
    ], axis=1)
    val_feats = scaler.transform(val_feats)
    val_probs = meta.predict_proba(val_feats)[:, 1]

    preds_meta = meta.predict(val_feats).astype(int)   # direct class predictions
    # Compute confusion matrix and derived counts
    # Expect val_member_labels to be an array-like of 0/1 ground-truth membership labels
    y_true = np.asarray(val_member_labels)
    y_pred = preds_meta
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    # Compute metrics
    f1 = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    try:
        auc_meta = roc_auc_score(y_true, val_probs)
    except Exception:
        auc_meta = float("nan")

    print("\nValidation metrics (meta classifier):")
    print(f"  Samples: {len(y_true)}")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1 score: {f1:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall: {rec:.4f}")
    print(f"  AUC (probabilities): {auc_meta:.4f}")
    print(f"  Confusion matrix counts: TP={tp}, FP={fp}, TN={tn}, FN={fn}")
    print(f"  Probabilities: min={val_probs.min():.4f}, max={val_probs.max():.4f}, mean={val_probs.mean():.4f}")



Device: cuda

Validation set: 192 members out of 1000 total
Member ratio: 19.20%

Computing victim model scores on validation set...

Baseline Loss-threshold attack:
AUC: 0.5210
F1: 0.3317, Precision: 0.2052, Recall: 0.8646
Chosen threshold: -0.1710

--- Training shadow models ---

[Shadow 1/3]


 20%|██        | 51/250 [00:08<00:32,  6.16it/s]

{'loss': 0.4836, 'grad_norm': 6.302373886108398, 'learning_rate': 4e-05, 'epoch': 0.4}


 40%|████      | 101/250 [00:15<00:22,  6.71it/s]

{'loss': 0.3752, 'grad_norm': 8.656774520874023, 'learning_rate': 3e-05, 'epoch': 0.8}


...

{'loss': 0.2938, 'grad_norm': 0.5463684797286987, 'learning_rate': 2e-05, 'epoch': 1.2}


 80%|████████  | 201/250 [00:28<00:07,  6.53it/s]

{'loss': 0.1907, 'grad_norm': 0.30781418085098267, 'learning_rate': 1e-05, 'epoch': 1.6}


100%|██████████| 250/250 [00:36<00:00,  6.93it/s]


{'loss': 0.221, 'grad_norm': 4.381056785583496, 'learning_rate': 0.0, 'epoch': 2.0}
{'train_runtime': 36.0461, 'train_samples_per_second': 55.485, 'train_steps_per_second': 6.936, 'train_loss': 0.2975984687805176, 'epoch': 2.0}
 Shadow 2: members=1000, nonmembers=1000

Shadow training stats:
Total samples: 6000, Members: 3000, Non-members: 3000
Feature shape: (6000, 4)
Meta-classifier trained on shadow data.

Evaluating meta-classifier on validation set...

Validation metrics (meta classifier):
  Samples: 1000
  Accuracy: 0.7120
  F1 score: 0.2000
  Precision: 0.2143
  Recall: 0.1875
  AUC (probabilities): 0.5186
  Confusion matrix counts: TP=36, FP=132, TN=676, FN=156
  Probabilities: min=0.2880, max=0.5094, mean=0.4626


Do not save from shadow models

In [4]:
# # --- APPLY TO EVALUATION SET ---
# print("\nComputing victim model scores on evaluation set...")
# eval_df = pd.read_csv(shadow_pool)
# eval_texts = eval_df["text"].tolist()
# eval_labels = eval_df["label"].astype(int).tolist()
# eval_scores = compute_scores(model, eval_texts, eval_labels)
# eval_loss_score = -eval_scores['neg_log_loss']

# if train_shadows:
#     eval_feats = np.stack([
#         eval_scores['max_prob'],
#         eval_loss_score,
#         eval_scores['entropy'],
#         eval_scores['margin']
#     ], axis=1)
#     eval_feats = scaler.transform(eval_feats)
#     eval_probs = meta.predict_proba(eval_feats)[:, 1]
#     eval_preds = meta.predict(eval_feats).astype(int)   # direct class predictions
# else:
#     eval_preds = (eval_loss_score >= best_thr).astype(int)


# # Write predictions to file
# with open(output_path, "w") as f:
#     for i, p in enumerate(eval_preds):
#         f.write(f"{i} {int(p)}\n")

# print(f"\nDone! Predictions saved to {output_path}")
# print(f"Evaluation predictions: {eval_preds.sum()} members, {len(eval_preds)-eval_preds.sum()} non-members")

In [5]:
# # A1 - preview file content
# with open("mia_lm_results.txt","r") as f:
#     lines = [l.strip() for l in f if l.strip()]
# print("Total lines in file:", len(lines))
# print("First 20 lines:")
# print("\n".join(lines[:20]))

# # A2 - count how many 1s and 0s were predicted
# ones = sum(1 for l in lines if l.split()[1] == '1')
# zeros = len(lines) - ones
# print(f"Predicted members (1): {ones}")
# print(f"Predicted non-members (0): {zeros}")
# print(f"Member ratio: {ones/len(lines):.4f}")